In [44]:
"""
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
VENV_DIR = ROOT / ".venv"
PYTHON_BIN = VENV_DIR / "bin" / "python"

if not VENV_DIR.exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)

subprocess.run([str(PYTHON_BIN), "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([str(PYTHON_BIN), "-m", "pip", "install", "jupyter", "ipykernel", "pandas", "pyarrow", "scikit-learn", "xgboost"], check=True)
subprocess.run([str(PYTHON_BIN), "-m", "ipykernel", "install", "--user", "--name", "songgraph-xgboost", "--display-name", "SongGraph XGBoost"], check=True)

print(VENV_DIR)
"""

'\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nROOT = Path.cwd()\nVENV_DIR = ROOT / ".venv"\nPYTHON_BIN = VENV_DIR / "bin" / "python"\n\nif not VENV_DIR.exists():\n    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)\n\nsubprocess.run([str(PYTHON_BIN), "-m", "pip", "install", "--upgrade", "pip"], check=True)\nsubprocess.run([str(PYTHON_BIN), "-m", "pip", "install", "jupyter", "ipykernel", "pandas", "pyarrow", "scikit-learn", "xgboost"], check=True)\nsubprocess.run([str(PYTHON_BIN), "-m", "ipykernel", "install", "--user", "--name", "songgraph-xgboost", "--display-name", "SongGraph XGBoost"], check=True)\n\nprint(VENV_DIR)\n'

In [45]:
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier

ROOT = Path.cwd()
DATA_PATH = ROOT / "valence_final.parquet"
#DATA_PATH = ROOT / "energy_final.parquet"

build_info = xgb.build_info()
cuda_built = str(build_info.get("USE_CUDA", "OFF")).upper() in {"ON", "TRUE", "1"}

try:
    nvidia_smi = subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )
    cuda_visible = nvidia_smi.returncode == 0
except FileNotFoundError:
    cuda_visible = False

XGB_DEVICE = "cuda" if cuda_built and cuda_visible else "cpu"
XGB_DEVICE


'cpu'

In [46]:
TARGET_COLUMNS = [
    "key",
    "mode",
]

FEATURE_POOL = [
    "valence",
    "loudness",
    "danceability",
    "energy",
    "tempo",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "key",
    "mode",
    "valence_missing",
    "loudness_missing",
    "danceability_missing",
    "energy_missing",
    "tempo_missing",
    "acousticness_missing",
    "instrumentalness_missing",
    "liveness_missing",
    "speechiness_missing",
    "key_missing",
    "mode_missing",
]

TARGET = "mode"
feature_columns = [column for column in FEATURE_POOL if column != TARGET]
OUTPUT_PATH = ROOT / f"{TARGET}_final.parquet"

MODEL_PARAMS = {
    "n_estimators": 600,
    "max_depth": 6,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "gamma": 0.0,
    "tree_method": "hist",
    "device": XGB_DEVICE,
    "random_state": 42,
    "n_jobs": -1,
}

if TARGET == "mode":
    MODEL_PARAMS["objective"] = "binary:logistic"
    MODEL_PARAMS["eval_metric"] = "logloss"
elif TARGET == "key":
    MODEL_PARAMS["objective"] = "multi:softprob"
    MODEL_PARAMS["num_class"] = 12
    MODEL_PARAMS["eval_metric"] = "mlogloss"

TARGET


'mode'

In [47]:
columns_to_load = sorted(set(FEATURE_POOL + ["track_id", "artist_name", "track_name"]))
df = pd.read_parquet(DATA_PATH, columns=columns_to_load)
float64_columns = df.select_dtypes(include=["float64"]).columns
df[float64_columns] = df[float64_columns].astype("float32")
df.shape


(3374899, 25)

In [48]:
df[TARGET_COLUMNS].isna().sum().sort_values(ascending=False)


key     446161
mode    446156
dtype: int64

In [49]:
observed_df = df[df[TARGET].notna()].copy()
missing_df = df[df[TARGET].isna()].copy()

X_observed = observed_df[feature_columns]
y_observed = observed_df[TARGET].astype("int32")
X_missing = missing_df[feature_columns]

X_train, X_test, y_train, y_test = train_test_split(
    X_observed,
    y_observed,
    test_size=0.2,
    random_state=42,
)

print(TARGET)
print(len(feature_columns))
print(len(observed_df))
print(len(missing_df))
print(len(X_train))
print(len(X_test))


mode
21
2928743
446156
2342994
585749


In [50]:
model = XGBClassifier(**MODEL_PARAMS)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    estimator=model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "f1_macro": "f1_macro",
    },
    n_jobs=-1,
    return_train_score=False,
)

cv_metrics = {
    "val_accuracy": cv_results["test_accuracy"].mean(),
    "val_f1_macro": cv_results["test_f1_macro"].mean(),
}

cv_metrics


{'val_accuracy': np.float64(0.7010517307851606),
 'val_f1_macro': np.float64(0.6380467843728359)}

In [51]:
model = XGBClassifier(**MODEL_PARAMS)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

test_metrics = {
    "accuracy": accuracy_score(y_test, predictions),
    "f1_macro": f1_score(y_test, predictions, average="macro"),
}

test_metrics


{'accuracy': 0.7020635118455174, 'f1_macro': 0.6392151275374551}

In [52]:
final_model = XGBClassifier(**MODEL_PARAMS)
final_model.fit(X_observed, y_observed)

predictions = final_model.predict(X_missing)

imputed_rows = missing_df[["track_id", "artist_name", "track_name"]].copy()
imputed_rows[TARGET] = predictions

imputed_rows.head()


,track_id,artist_name,track_name,mode
3,NaN,yungmanny,waldo,1
9,NaN,creature feature,bound and gagged,1
20,NaN,three 6 mafia,south memphis bitch,1
23,NaN,roxy music,for your pleasure,1
28,NaN,stick figure,angels among us,1


In [53]:
print(imputed_rows[TARGET].describe())
print("---")
print(imputed_rows[TARGET].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
print("---")
print(
imputed_rows[TARGET].min(),
imputed_rows[TARGET].max(),
imputed_rows[TARGET].mean(),
imputed_rows[TARGET].std(),
)
print("---")
print(imputed_rows[TARGET].nunique())

count    446156.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: mode, dtype: float64
---
0.01    1.0
0.05    1.0
0.25    1.0
0.50    1.0
0.75    1.0
0.95    1.0
0.99    1.0
Name: mode, dtype: float64
---
1 1 1.0 0.0
---
1


In [54]:
completed_rows = df[df[TARGET].notna()].copy()

print(completed_rows[TARGET].describe())
print("---")
print(completed_rows[TARGET].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
print("---")
print(
completed_rows[TARGET].min(),
completed_rows[TARGET].max(),
completed_rows[TARGET].mean(),
completed_rows[TARGET].std(),
)
print("---")
print(completed_rows[TARGET].nunique())

count    2.928743e+06
mean     6.425658e-01
std      4.792443e-01
min      0.000000e+00
25%      0.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: mode, dtype: float64
---
0.01    0.0
0.05    0.0
0.25    0.0
0.50    1.0
0.75    1.0
0.95    1.0
0.99    1.0
Name: mode, dtype: float64
---
0.0 1.0 0.6425658 0.4792443
---
2


In [55]:
missing_idx = df.index[df[TARGET].isna()]
df.loc[missing_idx, TARGET] = predictions
df[[TARGET]].isna().sum()


mode    0
dtype: int64

In [56]:
df.to_parquet(OUTPUT_PATH)
OUTPUT_PATH


PosixPath('/Users/saake/SongGraph/data_cleaning/mode_final.parquet')